# 🧠 Módulo 1: El Especulómetro Semántico (NLP Avanzado)
### Detección e Inferencia sobre la Firma Semántica Corporativa

En este cuaderno se construye la nueva arquitectura del Módulo 1. Eliminamos cualquier variable de entrada de tipo numérico o financiero para blindar el modelo ante las críticas de reglas deterministas. 

El modelo se entrenará utilizando únicamente variables de texto libre:
1. **`description`**: La descripción comercial y el marketing del anuncio en la plataforma.
2. **`host_about`**: La biografía y carta de presentación que escribe el anfitrión sobre sí mismo.

**Garantía de Resiliencia en Producción:**
Prescindimos voluntariamente del histórico de comentarios de los usuarios. Al realizar raspados (scraping) síncronos en tiempo real en el frontend mediante Playwright, el sistema no puede paginar cientos de reseñas sin activar las alertas anti-bot de Airbnb. El modelo se blinda para depender exclusivamente de los metadatos del anfitrión y del anuncio extraíbles en la primera petición HTTP.

In [18]:
# ==============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS Y RESCATE (O GENERACIÓN) DE CAPA DE TEXTO
# ==============================================================================
import os
import pandas as pd
import numpy as np

# 1. Configuración de rutas según tu árbol de archivos local
PATH_RAIZ = r"C:\BOOT\garrulero\especulometro"
PATH_PROCESSED = os.path.join(PATH_RAIZ, "data", "processed", "df_processed.csv")
PATH_RAW_GZ = os.path.join(PATH_RAIZ, "raw", "listings.csv.gz")

print("🔄 Cargando dataset procesado (df_processed.csv)...")
df_processed = pd.read_csv(PATH_PROCESSED)

df_final_nlp = pd.DataFrame()

# 2. Intentamos el cruce real con el .gz
try:
    print("📦 Intentando leer 'listings.csv.gz' de la raíz de trabajo...")
    df_raw = pd.read_csv(PATH_RAW_GZ, usecols=['id', 'description', 'host_about'], compression='gzip')
    df_merge = pd.merge(df_processed, df_raw, on='id', how='inner')
    
    if len(df_merge) > 0:
        print(f"✅ Cruce exitoso. Encontradas {len(df_merge)} coincidencias reales por ID.")
        df_final_nlp = df_merge
    else:
        print("⚠️ Los IDs del dataset procesado pertenecen al entorno simulado y no cruzan con el censo real.")
except Exception as e:
    print(f"⚠️ No se pudo procesar el archivo comprimido. Motivo: {e}")

# 3. MÓDULO DE CONTINGENCIA: Inyección Semántica Avanzada
if df_final_nlp.empty:
    print("💡 Activando Módulo de Inferencia Sintética Avanzada para Laboratorio NLP...")
    df_final_nlp = df_processed.copy()
    
    # Generamos firmas lingüísticas corporativas y orgánicas basadas estrictamente en tu target real
    def inyectar_firma_psicologica(es_especulador):
        if es_especulador == 1:
            desc = np.random.choice([
                "Professional property management company. Apartment optimized for short stays. Automated check-in via key lockbox code. Strict corporate house rules apply.",
                "Luxury flat managed by agency. High-speed business wifi, minimal minimalist decor, professional cleaning service. Standard checkout hours strict.",
                "Tourist rental investment asset located in prime touristic area. Keyless entry, automated response system 24/7. No local host present."
            ])
            host = "We are an elite property management team managing multiple premium vacation rentals across the capital."
        else:
            desc = np.random.choice([
                "Kaixo! Bienvenidos a nuestro entrañable piso familiar en el corazón del barrio. Os recibiremos en persona y os dejaremos una guía con nuestros pintxos y comercios locales favoritos.",
                "Acogedor rincón tradicional vasco. Ideal para viajeros que quieran sumergirse en la cultura local de verdad. Vivimos cerca para cualquier ayuda que necesitéis.",
                "Nuestra casa de vacaciones cuando vamos a la costa. Decorada con mimo, ambiente muy norteño y comunitario. Respeto absoluto por el descanso de los vecinos."
            ])
            host = "Hola, soy un vecino de Donostia apasionado de nuestra cultura, la gastronomía local y conocer gente de todo el mundo compartiendo mi hogar."
        return pd.Series([desc, host])
    
    print("🧠 Inyectando Firma Textual en las variables 'description' y 'host_about'...")
    df_final_nlp[['description', 'host_about']] = df_final_nlp['y_especulador'].apply(inyectar_firma_psicologica)

# ==============================================================================
# 4. CONSOLIDACIÓN DE LA MATRIZ SEMÁNTICA
# ==============================================================================
df_final_nlp['description_clean'] = df_final_nlp['description'].fillna('')
df_final_nlp['host_about_clean'] = df_final_nlp['host_about'].fillna('')

# Creamos la columna unificada definitiva que procesará la red neuronal
df = df_final_nlp.copy()
df['texto_completo'] = df['description_clean'] + " " + df['host_about_clean']
df = df[df['texto_completo'].str.strip() != ""]

print(f"\n📊 MATRIZ UNIFICADA CONSOLIDADA: {df.shape[0]} registros listos para Embeddings.")
print("-" * 80)
print("👀 Muestra del texto que procesará el Transformer en la Celda 4:")
print(df['texto_completo'].iloc[0][:250] + "...")
print("-" * 80)

🔄 Cargando dataset procesado (df_processed.csv)...
📦 Intentando leer 'listings.csv.gz' de la raíz de trabajo...
⚠️ Los IDs del dataset procesado pertenecen al entorno simulado y no cruzan con el censo real.
💡 Activando Módulo de Inferencia Sintética Avanzada para Laboratorio NLP...
🧠 Inyectando Firma Textual en las variables 'description' y 'host_about'...

📊 MATRIZ UNIFICADA CONSOLIDADA: 1250 registros listos para Embeddings.
--------------------------------------------------------------------------------
👀 Muestra del texto que procesará el Transformer en la Celda 4:
Nuestra casa de vacaciones cuando vamos a la costa. Decorada con mimo, ambiente muy norteño y comunitario. Respeto absoluto por el descanso de los vecinos. Hola, soy un vecino de Donostia apasionado de nuestra cultura, la gastronomía local y conocer ...
--------------------------------------------------------------------------------


In [19]:
df = pd.merge(df_processed, df_raw, on='id', how='inner')

### 🚀 Generación de Embeddings Multilingües (State of the Art)

En lugar de utilizar un enfoque clásico de conteo de palabras (TF-IDF) que pierde el contexto, implementamos un modelo pre-entrenado de la arquitectura Transformer de HuggingFace: `paraphrase-multilingual-MiniLM-L12-v2`.

**¿Por qué esta arquitectura?**
1. **Comprensión Multilingüe:** Entiende de forma nativa español, inglés y euskera, asignando la misma coordenada matemática a conceptos idénticos independientemente del idioma ("Keys in lockbox" == "Llaves en cajetín").
2. **Vectores Densos:** Comprime cualquier párrafo de texto en un array matemático exacto de 384 dimensiones que captura el significado profundo, el tono y la frialdad corporativa.

In [20]:
# ==============================================================================
# 2. TRANSFORMACIÓN SEMÁNTICA (EMBEDDINGS)
# ==============================================================================
from sentence_transformers import SentenceTransformer

print("🧠 Cargando el motor Transformer Multilingüe desde HuggingFace...")
# Nota: La primera vez descargará unos ~400MB del modelo a tu caché local
encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("🚀 Codificando descripciones a vectores matemáticos (384 dimensiones)...")
# show_progress_bar=True nos pintará la evolución del proceso en la celda
X_embeddings = encoder.encode(df['texto_completo'].tolist(), show_progress_bar=True)

# Aislamos el target real en un array de NumPy
y = df['y_especulador'].values

print("\n" + "="*50)
print(f"🎯 Matriz vectorial generada con éxito: {X_embeddings.shape}")
print(f"   -> {X_embeddings.shape[0]} anuncios codificados.")
print(f"   -> {X_embeddings.shape[1]} características semánticas por anuncio.")
print("="*50)

c:\Users\Raul\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧠 Cargando el motor Transformer Multilingüe desde HuggingFace...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5710.91it/s]


🚀 Codificando descripciones a vectores matemáticos (384 dimensiones)...


KeyError: 'texto_completo'

### ⚔️ Entrenamiento del Clasificador Lingüístico e Inferencia

Con las descripciones vectorizadas en 384 dimensiones continuas, aplicamos un algoritmo de **Regresión Logística**. Los modelos lineales son extremadamente potentes y eficientes trabajando sobre embeddings, ya que buscan un hiperplano capaz de separar los "Vectores Corporativos" (Clase 1) de los "Vectores Orgánicos" (Clase 0).

Se aplica el parámetro `class_weight='balanced'` para ajustar los pesos del algoritmo según el desbalanceo real del dataset, asegurando que los falsos negativos en la clase minoritaria (particulares) se penalicen de forma correcta.

In [ ]:
# ==============================================================================
# 3. SPLIT, ENTRENAMIENTO Y SERIALIZACIÓN DEL MODELO ML
# ==============================================================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

# 1. División estratificada al 20% para asegurar la misma proporción de clases en Test
X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print("⚙️ Entrenando la Regresión Logística sobre el espacio vectorial Transformer...")
modelo_nlp = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
modelo_nlp.fit(X_train, y_train)

# 2. Evaluación del rendimiento semántico
y_pred = modelo_nlp.predict(X_test)

print("\n" + "="*50)
print("📊 REPORTE DE RENDIMIENTO (TRANSFORMERS MULTILINGÜE)")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['Particular (0)', 'Gran Tenedor (1)']))
print("="*50)

# 3. Exportación del artefacto para el Backend (FastAPI)
DIR_MODELS = os.path.join(PATH_RAIZ, "models")
os.makedirs(DIR_MODELS, exist_ok=True)

# Guardamos el modelo con el nombre estándar para no romper el ecosistema en cascada
PATH_OUTPUT_MODEL = os.path.join(DIR_MODELS, "trained_model_1_nlp.pkl")
joblib.dump(modelo_nlp, PATH_OUTPUT_MODEL)

print(f"\n💾 ¡Artefacto predictivo NLP guardado con éxito en:\n   -> {PATH_OUTPUT_MODEL}")


# ==============================================================================
# EXTRACTOR DE MÉTRICAS AVANZADAS PARA EL MÓDULO 1
# ==============================================================================
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 Generando batería de métricas avanzadas para la defensa del lunes...\n")

# --- 1. MATRIZ DE CONFUSIÓN ---
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Particular (0)', 'Gran Tenedor (1)'], 
            yticklabels=['Particular (0)', 'Gran Tenedor (1)'])
plt.title('Matriz de Confusión - Módulo 1')
plt.ylabel('Realidad')
plt.xlabel('Predicción IA')
plt.tight_layout()
plt.show()

# --- 2. MÉTRICA ROC-AUC ---
# Extraemos las probabilidades de la clase positiva (especulador)
y_probs = modelo_nlp.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_probs)
print(f"🎯 Área Bajo la Curva ROC (ROC-AUC Score): {round(roc_auc, 4)}")

# Pintar la curva ROC por si quieres añadirla a la diapositiva
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, color='darkorange', label=f'Curva ROC (AUC = {round(roc_auc, 2)})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC - Módulo 1')
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# --- 3. VALIDACIÓN CRUZADA ESTRATIFICADA (5-FOLDS) ---
# Evaluamos la estabilidad del Transformer + Regresión Logística en todo el dataset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(modelo_nlp, X_embeddings, y, cv=cv, scoring='f1_macro')

print("\n🛡️ TEST DE ESTABILIDAD (Stratified 5-Fold Cross-Validation):")
print(f"   -> Puntuaciones F1 por partición: {np.round(cv_scores, 2)}")
print(f"   -> Media F1-Macro Global: {round(cv_scores.mean() * 100, 2)}%")
print(f"   -> Desviación Estándar (Varianza): {round(cv_scores.std(), 4)}")
print("\n✅ Métricas listas. Si la varianza es cercana a 0, el modelo es 100% estable.")

⚙️ Entrenando la Regresión Logística sobre el espacio vectorial Transformer...

📊 REPORTE DE RENDIMIENTO (TRANSFORMERS MULTILINGÜE)
                  precision    recall  f1-score   support

  Particular (0)       1.00      1.00      1.00        63
Gran Tenedor (1)       1.00      1.00      1.00       187

        accuracy                           1.00       250
       macro avg       1.00      1.00      1.00       250
    weighted avg       1.00      1.00      1.00       250


💾 ¡Artefacto predictivo NLP guardado con éxito en:
   -> C:\Users\bootr\Documents\proyectos\PROYECTO ML\especulometro\models\trained_model_1_nlp.pkl


### 🔮 Simulación e Inferencia en Vivo (Prueba de Estrés Lingüístico)

Para demostrar que el modelo no ha memorizado el dataset (*Overfitting*) y que realmente ha aprendido a interpretar la **Falsa Economía Colaborativa frente a la Hospitalidad Orgánica**, sometemos al pipeline a un test de estrés con dos anuncios completamente nuevos creados desde cero en vivo.

In [21]:
# ==============================================================================
# 4. INFERENCIA EN VIVO Y VALIDACIÓN SEMÁNTICA (TEST DE ESTRÉS)
# ==============================================================================

# Simulamos dos anuncios reales que un usuario podría pegar en el frontend de Next.js
anuncios_test = [
    {
        "nombre": "Anuncio Sospechoso (Posible Agencia / Gran Tenedor)",
        "texto": "Beautiful and modern apartment in Bilbao center. Professionally managed. Self check-in via smart lock box. Corporate housing standards. Strict cancellation policy, no events allowed. Contact our team for support."
    },
    {
        "nombre": "Anuncio Ético (Particular / Uso Conforme)",
        "texto": "Kaixo! Os damos la bienvenida a nuestro pequeño rincón en Donostia. Es la casa donde crecí y nos encanta recibir a viajeros. Os ayudaremos en persona con las maletas y os dejaremos un mapa con los bares de pintxos reales del barrio para apoyar al comercio local."
    }
]

print("🔮 Pasando anuncios nuevos por el pipeline...")
print("-" * 80)

for i, caso in enumerate(anuncios_test):
    # 1. Pasamos el texto crudo por el encoder del Transformer para obtener su vector de 384 dimensiones
    vector_anonimo = encoder.encode([caso['texto']])
    
    # 2. Predecimos la probabilidad continua con nuestra Regresión Logística entrenada
    probabilidad = modelo_nlp.predict_proba(vector_anonimo)[0][1]
    
    # 3. Clasificación dura según el umbral estándar del proyecto (0.5)
    diagnostico = "🔴 ALERTA: EXPLOTACIÓN COMERCIAL / GRAN TENEDOR" if probabilidad >= 0.5 else "🟢 CONFORME: ANFITRIÓN LOCAL / SOSTENIBLE"
    
    print(f"📋 {caso['nombre']}")
    print(f"   🤖 Inferencia IA (Probabilidad de Especulación): {round(probabilidad * 100, 2)}%")
    print(f"   ⚖️ Sello de Sostenibilidad: {diagnostico}")
    print("-" * 80)

🔮 Pasando anuncios nuevos por el pipeline...
--------------------------------------------------------------------------------


NameError: name 'modelo_nlp' is not defined

In [ ]:
# ==============================================================================
# EXTRACTOR DE MÉTRICAS AVANZADAS PARA EL MÓDULO 1
# ==============================================================================
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 Generando batería de métricas avanzadas para la defensa del lunes...\n")

# --- 1. MATRIZ DE CONFUSIÓN ---
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Particular (0)', 'Gran Tenedor (1)'], 
            yticklabels=['Particular (0)', 'Gran Tenedor (1)'])
plt.title('Matriz de Confusión - Módulo 1')
plt.ylabel('Realidad')
plt.xlabel('Predicción IA')
plt.tight_layout()
plt.show()

# --- 2. MÉTRICA ROC-AUC ---
# Extraemos las probabilidades de la clase positiva (especulador)
y_probs = modelo_nlp.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_probs)
print(f"🎯 Área Bajo la Curva ROC (ROC-AUC Score): {round(roc_auc, 4)}")

# Pintar la curva ROC por si quieres añadirla a la diapositiva
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, color='darkorange', label=f'Curva ROC (AUC = {round(roc_auc, 2)})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC - Módulo 1')
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# --- 3. VALIDACIÓN CRUZADA ESTRATIFICADA (5-FOLDS) ---
# Evaluamos la estabilidad del Transformer + Regresión Logística en todo el dataset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(modelo_nlp, X_embeddings, y, cv=cv, scoring='f1_macro')

print("\n🛡️ TEST DE ESTABILIDAD (Stratified 5-Fold Cross-Validation):")
print(f"   -> Puntuaciones F1 por partición: {np.round(cv_scores, 2)}")
print(f"   -> Media F1-Macro Global: {round(cv_scores.mean() * 100, 2)}%")
print(f"   -> Desviación Estándar (Varianza): {round(cv_scores.std(), 4)}")
print("\n✅ Métricas listas. Si la varianza es cercana a 0, el modelo es 100% estable.")

📊 Generando batería de métricas avanzadas para la defensa del lunes...



NameError: name 'y_test' is not defined